# CAIA cohort - figure generation

Copy of `PROFILE/generate_figures.ipynb` with the `BASE` path repointed at `survival_analysis/CAIA/local_runs`.  Figure code, color palette, and category mapping are unchanged - they live in `helpers/plotting.py` so both cohorts use the same rcParams and CATEGORY_COLORS.

> _Skipped for CAIA: this panel needs `cox_pgs_adjusted.csv`, which the CAIA pipeline does not produce (no PGS data, no PSA/Testosterone in cohort)._

# Generate platinum-arm figures

End-to-end driver that builds the four platinum-arm figures from the survival pipeline outputs:

1. **Paired volcano** (`paired_volcano_platinum.{png,pdf}`) — univariate Cox at landmarks 0d and +90d, colored by clinical category.
2. **Grouped AUC barplot** (`auc_grouped_barplot_platinum.{png,pdf}`) — test mean AUC(t) for Elastic-Net Cox, XGBoost Survival, and Dynamic DeepHit at both landmarks.
3. **2×2 importance grid** (`model_importance_platinum.{png,pdf}`) — top-15 features by Cox log-HR coefficient (top row) and XGBoost gain (bottom row) at both landmarks.
4. **PGS interpretation** (`pgs_interpretation_platinum.{png,pdf}`) — PGS-only vs lab-adjusted PGS HRs for Testosterone and PSA at both landmarks, classified into mediated / independent / suppressor / no-signal regimes.

Each figure section is independent once the shared `Imports`, `Paths`, and `Clinical category mapping` cells have been run.

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Paths

`BASE` points at the directory containing `cox/landmark_{lm}/...`, `xgboost/landmark_{lm}/...`, and `deephit/landmark_{lm}/...` outputs. Override for local copies.

In [ ]:
BASE = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/"
    "survival_analysis/CAIA/local_runs"
)
OUT_DIR = Path("./figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS = [0, 90]
TOP_N = 15  # number of features shown per importance panel

## Clinical category mapping

40 labs in six buckets. `Body height` is dropped due to data-quality concerns. PT is folded into LFT since it tracks hepatic synthetic function. The same palette is reused across all three figures.

In [ ]:
CBC = {
    "WBC", "RBC", "Hemoglobin", "Hematocrit",
    "MCV", "MCH", "MCHC", "RDW", "Platelets",
    "Neutrophils absolute", "Lymphocytes absolute",
    "Monocytes absolute", "Eosinophils absolute", "Basophils absolute",
}
CMP = {"Sodium", "Potassium", "Chloride", "CO2",
       "BUN", "Creatinine", "Glucose", "Calcium"}
LFT = {"ALT", "AST", "Alkaline phosphatase",
       "Total bilirubin", "Direct bilirubin",
       "Albumin", "Globulin", "Total protein", "PT"}
VITALS = {"Body weight", "Body temperature", "Heart rate",
          "Respiratory rate", "Systolic blood pressure",
          "Diastolic blood pressure"}
ANDROGEN = {"PSA", "Testosterone"}
OTHER = {"TSH"}
DROP = {"Body height"}

CATEGORY_MAP = {}
for label, members in [
    ("CBC", CBC), ("CMP", CMP), ("LFT", LFT),
    ("Vitals", VITALS), ("Androgen axis", ANDROGEN), ("Other", OTHER),
]:
    for m in members:
        CATEGORY_MAP[m] = label

DRAW_ORDER = ["Other", "Vitals", "CMP", "LFT", "CBC", "Androgen axis"]
LEGEND_ORDER = ["Androgen axis", "CBC", "LFT", "CMP", "Vitals", "Other"]

CATEGORY_COLORS = {
    "Androgen axis": "#8e1c2b",
    "CBC":           "#16a085",
    "LFT":           "#e67e22",
    "CMP":           "#7d3c98",
    "Vitals":        "#5d6d7e",
    "Other":         "#95a5a6",
}
NS_COLOR = "#d5d8dc"


# CAIA cohort: lab names are LOINC-style (e.g. 'Glucose [Mass/volume] in
# Serum or Plasma').  Route them through helpers.plotting.canonicalize_lab_name
# so they land in the same CBC/CMP/LFT/Vitals/Androgen/Other buckets as PROFILE.
import sys
sys.path.insert(0, str(Path('/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/survival_analysis')))
from helpers.plotting import canonicalize_lab_name

def assign_category(lab_name: str) -> str:
    return CATEGORY_MAP.get(canonicalize_lab_name(lab_name), "Other")


def format_label(lab_name, feature_stat):
    if not feature_stat or pd.isna(feature_stat) or feature_stat == "":
        return lab_name
    return f"{lab_name} ({feature_stat})"


def parse_feature(name):
    """Split 'LAB_NAME__stat' into (lab_name, stat). Returns (name, '') if no '__'."""
    if "__" in name:
        lab, stat = name.split("__", 1)
        return lab, stat
    return name, ""

## Figure 1 — Paired volcano (univariate Cox)

Each dot is one (lab × feature_stat) pair. ns features (q ≥ 0.05) are faded gray; q-significant features are colored by their clinical category. Three narrative beats highlighted on the figure:

- **Testosterone** is a key indicator at both landmarks
- **PSA** becomes significant at +90d
- **CBC + LFT** signals (general health markers) emerge by +90d

Source: `cox/landmark_{lm}/both/cox_agg_univariate_nobs_adjusted.csv`.

In [ ]:
# ----------------------------- labeling knobs ---------------------------
TOP_K_PER_PANEL = 6
ALWAYS_LABEL = {"Hemoglobin", "Albumin", "Alkaline phosphatase"}
LABEL_FORMAT = "{lab} ({stat})"
LABEL_FONTSIZE = 8.5
MIN_LABEL_SPACING_NLP = 0.55
PANEL_XLIM = (-1.5, 1.5)


def q_threshold_neglog10p(sub):
    """y-value (-log10 p) at which q just crosses 0.05; None if no q-sig features."""
    sig = sub.loc[sub["q_value"] < 0.05, "p_value"]
    if sig.empty:
        return None
    cutoff_p = float(sig.max())
    return float(-np.log10(max(cutoff_p, 1e-300)))


def _auto_label(ax, sub, *, top_k, always_label, label_format):
    """Stack labels at the left/right panel edges with leader lines.

    Selection rules:
      - Androgen axis: every significant (lab × stat) is labeled (no dedup).
      - Other categories: dedupe to the most-significant stat per lab,
        then include `always_label` whitelist + top_k by p-value.
    """
    sig = sub.loc[sub["sig"]].copy()
    if sig.empty:
        return

    androgen_rows = sig.loc[sig["category"] == "Androgen axis"]

    non_andro = sig.loc[sig["category"] != "Androgen axis"]
    best = (non_andro.sort_values("p_value")
                     .drop_duplicates("lab_name", keep="first"))
    always_sig = best.loc[best["lab_name"].isin(always_label)]
    extra = (best.loc[~best["lab_name"].isin(always_label)].head(top_k))
    non_andro_label = pd.concat([always_sig, extra]).drop_duplicates("lab_name")

    to_label = pd.concat([androgen_rows, non_andro_label]).drop_duplicates(
        subset=["lab_name", "feature_stat"])
    if to_label.empty:
        return

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xspan = xlim[1] - xlim[0]
    yspan = ylim[1] - ylim[0]

    def _stack(side_df, side):
        if side_df.empty:
            return
        side_df = side_df.sort_values("neglog10p", ascending=False)
        if side == "left":
            label_x = xlim[0] + 0.04 * xspan
            ha = "left"
        else:
            label_x = xlim[1] - 0.04 * xspan
            ha = "right"
        last_y = None
        for _, r in side_df.iterrows():
            target_y = r["neglog10p"]
            if last_y is not None and target_y > last_y - MIN_LABEL_SPACING_NLP:
                target_y = last_y - MIN_LABEL_SPACING_NLP
            if target_y < ylim[0] + 0.05 * yspan:
                continue
            last_y = target_y
            text = label_format.format(lab=r["lab_name"],
                                       stat=r["feature_stat"])
            color = CATEGORY_COLORS[r["category"]]
            ax.annotate(
                text, xy=(r["coef_feature"], r["neglog10p"]),
                xytext=(label_x, target_y), textcoords="data",
                ha=ha, va="center",
                fontsize=LABEL_FONTSIZE, color=color, weight="semibold",
                arrowprops=dict(arrowstyle="-", lw=0.45, color="#95a5a6"),
                zorder=10,
            )

    left = to_label.loc[to_label["coef_feature"] < 0]
    right = to_label.loc[to_label["coef_feature"] >= 0]
    _stack(left, "left")
    _stack(right, "right")


def plot_volcano_panel(ax, sub, title):
    sub = sub.loc[~sub["lab_name"].isin(DROP)].copy()
    sub["category"] = sub["lab_name"].map(assign_category)
    sub["neglog10p"] = -np.log10(sub["p_value"].clip(lower=1e-300))
    sub["sig"] = sub["q_value"] < 0.05

    ns = sub.loc[~sub["sig"]]
    ax.scatter(ns["coef_feature"], ns["neglog10p"],
               s=20, color=NS_COLOR, alpha=0.45,
               edgecolor="none", zorder=1)

    for cat in DRAW_ORDER:
        cat_sig = sub.loc[sub["sig"] & (sub["category"] == cat)]
        if cat_sig.empty:
            continue
        is_hero = cat == "Androgen axis"
        ax.scatter(
            cat_sig["coef_feature"], cat_sig["neglog10p"],
            s=80 if is_hero else 40,
            color=CATEGORY_COLORS[cat],
            alpha=0.92,
            edgecolor="white", linewidth=0.6,
            zorder=5 if is_hero else 3,
        )

    ax.set_xlim(*PANEL_XLIM)
    y_max = float(sub["neglog10p"].max()) if not sub.empty else 5.0
    ax.set_ylim(-0.2, max(y_max * 1.10, 5.0))

    ax.axvline(0, color="grey", linestyle="-", linewidth=0.7, zorder=0)
    for x in (-0.5, 0.5):
        ax.axvline(x, color="grey", linestyle="--", linewidth=0.6,
                   alpha=0.7, zorder=0)
    q_y = q_threshold_neglog10p(sub)
    if q_y is not None:
        ax.axhline(q_y, color="black", linestyle=":", linewidth=0.9, zorder=0)

    _auto_label(ax, sub,
                top_k=TOP_K_PER_PANEL,
                always_label=ALWAYS_LABEL,
                label_format=LABEL_FORMAT)

    ax.set_xlabel("Cox log HR per SD")
    ax.set_ylabel(r"$-\log_{10}(p)$")
    ax.set_title(title, fontsize=12.5, weight="bold")

    n_tested = len(sub)
    n_sig = int(sub["sig"].sum())
    breakdown = (sub.loc[sub["sig"], "category"]
                 .value_counts()
                 .reindex([c for c in LEGEND_ORDER if c != "Other"], fill_value=0))
    short = {"Androgen axis": "Androgen", "CBC": "CBC", "LFT": "LFT",
             "CMP": "CMP", "Vitals": "Vitals"}
    bd_str = "  ".join(f"{short[c]} {int(n)}" for c, n in breakdown.items())
    ax.text(0.99, 0.02,
            f"{n_sig} / {n_tested} q<0.05   ·   {bd_str}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8.5, color="#5d6d7e", family="monospace")

In [ ]:
# Load both landmarks and tag with landmark_days (authoritative from path).
def _load_uni(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
    df = pd.read_csv(p)
    df["landmark_days"] = landmark
    return df

uni = pd.concat([_load_uni(lm) for lm in LANDMARKS], ignore_index=True)
uni = uni.loc[uni["endpoint"] == "platinum"].copy()
uni = uni.dropna(subset=["coef_feature", "p_value", "q_value"])
print(f"{len(uni)} (lab × stat) rows across landmarks "
      f"{sorted(uni['landmark_days'].unique())}")

# Filter unstable Cox estimates: |log HR| > 2 or CI spans > 2 orders of magnitude.
COEF_CAP = 2.0
CI_RATIO_CAP = 100
ci_ratio = uni["ci_upper"] / uni["ci_lower"]
mask = (uni["coef_feature"].abs() <= COEF_CAP) & (ci_ratio < CI_RATIO_CAP)
print(f"dropping {int((~mask).sum())} / {len(uni)} unstable rows")
uni = uni.loc[mask].copy()
print(f"{len(uni)} rows remaining")

panels = [(0, "0 days"), (90, "+90 days")]
fig, axes = plt.subplots(
    1, 2, figsize=(14, 6),
    sharex=True, sharey=False,
    gridspec_kw={"wspace": 0.08},
)
for ax, (lm, title) in zip(axes, panels):
    sub = uni.loc[uni["landmark_days"] == lm]
    if sub.empty:
        ax.text(0.5, 0.5, f"(no data for landmark = {lm}d)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_axis_off()
        continue
    plot_volcano_panel(ax, sub, title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
handles.append(Line2D([0], [0], marker="o", color="w",
                      markerfacecolor=NS_COLOR, markersize=8,
                      label="ns (q ≥ 0.05)"))
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"paired_volcano_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

## Figure 2 — Grouped AUC barplot

Mean IPCW cumulative/dynamic AUC(t) on the held-out test fold for the three multivariable models, with bars paired across landmarks so the cross-landmark comparison is visible at a glance.

Sources (one CSV per model × landmark):
- **Cox**     `cox/landmark_{lm}/both/cox_agg_multivariable_metrics.csv`        → `endpoint == "platinum"`, column `test_mean_auc_t`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_metrics.csv`         → `endpoint == "platinum"`, column `mean_auc_t`
- **DeepHit** `deephit/landmark_{lm}/PLATINUM/dynamic_deephit_metrics_PLATINUM.csv` → `event == "PLATINUM"`, column `mean_auc_t`

In [ ]:
def load_cox_auc(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable_metrics.csv"
    df = pd.read_csv(p)
    return float(df.loc[df["endpoint"] == "platinum"].iloc[0]["test_mean_auc_t"])


def load_xgb_auc(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_metrics.csv"
    df = pd.read_csv(p)
    return float(df.loc[df["endpoint"] == "platinum"].iloc[0]["mean_auc_t"])


def load_deephit_auc(landmark):
    p = BASE / "deephit" / f"landmark_{landmark}" / "PLATINUM" / "dynamic_deephit_metrics_PLATINUM.csv"
    df = pd.read_csv(p)
    return float(df.loc[df["event"] == "PLATINUM"].iloc[0]["mean_auc_t"])


MODEL_ORDER = ["Elastic-Net Cox", "XGBoost Survival"]  # Dynamic DeepHit excluded for now
AUC_LOADERS = {
    "Elastic-Net Cox":  load_cox_auc,
    "XGBoost Survival": load_xgb_auc,
}
MODEL_COLORS = {
    "Elastic-Net Cox":  "#4C72B0",
    "XGBoost Survival": "#B58900",
}

values = {m: [AUC_LOADERS[m](lm) for lm in LANDMARKS] for m in MODEL_ORDER}
n_models = len(MODEL_ORDER)
bar_width = 0.24
x_positions = np.arange(len(LANDMARKS))
offsets = (np.arange(n_models) - (n_models - 1) / 2) * bar_width

fig, ax = plt.subplots(figsize=(8.5, 5.5))
for i, m in enumerate(MODEL_ORDER):
    bar_x = x_positions + offsets[i]
    ax.bar(bar_x, values[m], bar_width,
           color=MODEL_COLORS[m],
           edgecolor="white", linewidth=0.8,
           label=m)
    for x, v in zip(bar_x, values[m]):
        ax.text(x, v + 0.008, f"{v:.3f}",
                ha="center", va="bottom",
                fontsize=10, weight="semibold",
                color=MODEL_COLORS[m])

all_vals = [v for vs in values.values() for v in vs]
y_top = min(1.0, max(all_vals) * 1.12)
ax.set_ylim(0.45, y_top)

ax.set_xticks(x_positions)
ax.set_xticklabels([f"{('+' if lm > 0 else '')}{lm} days" for lm in LANDMARKS])
ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9)
ax.set_ylabel("Test Mean AUC(t)")
ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="upper right", fontsize=10, ncol=1)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"auc_grouped_barplot_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

## Figure 3 — 2×2 model importance grid

Top features by importance for Elastic-Net Cox (signed log HR coefficient, top row) and XGBoost Survival (gain, bottom row), one column per landmark. AGE is excluded so the focus is on lab-derived features.

Sources:
- **Cox**     `cox/landmark_{lm}/both/cox_agg_multivariable.csv`              → `endpoint == "platinum"`, column `coef`
- **XGBoost** `xgboost/landmark_{lm}/both/landmark_xgboost_feature_importance.csv` → `endpoint == "platinum"`, column `gain`

In [ ]:
def load_cox_coefs(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[~df["is_age_covariate"].fillna(False).astype(bool)]
    df = df.loc[df["coef"].fillna(0) != 0]
    return df


def load_xgb_importance(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_feature_importance.csv"
    df = pd.read_csv(p)
    df = df.loc[df["endpoint"] == "platinum"].copy()
    df = df.loc[df["feature"].str.lower() != "age"]
    df = df.loc[df["gain"].fillna(0) > 0]
    parsed = df["feature"].apply(lambda s: pd.Series(parse_feature(s),
                                                     index=["lab_name", "feature_stat"]))
    df = pd.concat([df.reset_index(drop=True), parsed.reset_index(drop=True)],
                   axis=1)
    return df


def render_importance_panel(ax, df, *, kind, title):
    if df.empty:
        ax.text(0.5, 0.5, "(no features to display)",
                ha="center", va="center", transform=ax.transAxes,
                color="#7f8c8d")
        ax.set_title(title, fontsize=11, weight="bold")
        ax.set_axis_off()
        return

    df = df.copy()
    df["category"] = df["lab_name"].map(assign_category)

    if kind == "cox":
        df = df.reindex(df["coef"].abs().sort_values(ascending=False).index).head(TOP_N)
        df = df.iloc[::-1]
        values = df["coef"].to_numpy()
        xlabel = "log HR coefficient"
    else:  # xgb
        df = df.sort_values("gain", ascending=False).head(TOP_N).iloc[::-1]
        values = df["gain"].to_numpy()
        xlabel = "XGBoost gain"

    colors = [CATEGORY_COLORS[c] for c in df["category"]]
    y = np.arange(len(df))
    ax.barh(y, values, color=colors, edgecolor="white", linewidth=0.6)

    if kind == "cox":
        ax.axvline(0, color="black", linewidth=0.5, zorder=0)

    labels = [format_label(r["lab_name"], r["feature_stat"])
              for _, r in df.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11, weight="bold")


fig, axes = plt.subplots(2, 2, figsize=(13, 9.5), constrained_layout=True)
MODEL_ROWS = [
    ("cox", "Elastic-Net Cox", load_cox_coefs),
    ("xgb", "XGBoost Survival", load_xgb_importance),
]

for row, (kind, model_name, loader) in enumerate(MODEL_ROWS):
    for col, lm in enumerate(LANDMARKS):
        ax = axes[row, col]
        try:
            df = loader(lm)
        except FileNotFoundError as e:
            ax.text(0.5, 0.5, f"file not found:\n{e.filename}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="#c0392b")
            ax.set_axis_off()
            continue
        sign = "+" if lm > 0 else ""
        title = f"{model_name}  ·  {sign}{lm} days"
        render_importance_panel(ax, df, kind=kind, title=title)

handles = [Patch(facecolor=CATEGORY_COLORS[c], edgecolor="white", label=c)
           for c in LEGEND_ORDER]
fig.legend(handles=handles, loc="lower center",
           ncol=len(handles), bbox_to_anchor=(0.5, -0.04),
           fontsize=10)

for ext in ("png", "pdf"):
    out = OUT_DIR / f"model_importance_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()

> _Skipped for CAIA: this panel needs `cox_pgs_adjusted.csv`, which the CAIA pipeline does not produce (no PGS data, no PSA/Testosterone in cohort)._

## Figure 4 — PGS interpretation

For each PGS column, `cox_pgs_adjusted.py` fits **three** companion Cox models:

| Fit | Model | Question it answers |
|---|---|---|
| **PGS-only** | `Surv ~ AGE + PGS` | Does the PGS predict the endpoint *at all*? |
| **lab-only (baseline)** | `Surv ~ AGE + n_obs + LAB` | What does the lab marker do on its own? |
| **joint** | `Surv ~ AGE + PGS + n_obs + LAB` | What's left of each after adjusting for the other? |

This panel pairs the **PGS-only HR** (open circle) with the **joint PGS HR** (filled diamond, colored by regime) for each PGS at each (target lab × landmark). The thin connector shows the direction and magnitude of the attenuation when the lab is added to the model.

**Four regimes** (BH q < 0.05 on each side, computed per endpoint):

- 🟢 **Independent** — PGS-only sig **and** joint sig → PGS carries information beyond the lab marker.
- 🟠 **Mediated** — PGS-only sig **but** joint not → the lab absorbs the PGS signal; PGS adds nothing on top.
- 🟣 **Suppressor** — PGS-only **not** sig but joint **is** → rare; usually collinearity / variance suppression.
- ⚪ **No signal** — neither significant → PGS is null for this endpoint × lab combination.

Source: `cox/landmark_{lm}/both/cox_pgs_adjusted.csv` (one row per `endpoint × lab × PGS`). Endpoint is fixed to platinum to match the rest of this notebook.

In [ ]:
# CAIA has no cox_pgs_adjusted.csv (no PGS data, no PSA/Testosterone
# in the cohort).  Skipping the PGS-interpretation panel.
print('[skip] PGS-interpretation panel: not applicable to CAIA cohort')
if False:
    def load_pgs(landmark, endpoint="platinum"):
        p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_pgs_adjusted.csv"
        df = pd.read_csv(p)
        df = df.loc[df["endpoint"] == endpoint].copy()
        df["landmark_days"] = landmark
        return df
    
    
    REGIME_COLORS = {
        "independent": "#1abc9c",
        "mediated":    "#e67e22",
        "suppressor":  "#7d3c98",
        "no_signal":   "#bdc3c7",
    }
    REGIME_ORDER = ["independent", "mediated", "suppressor", "no_signal"]
    
    
    def classify_regime(row, alpha=0.05):
        q_only = row.get("q_value_pgs_only", np.nan)
        q_joint = row.get("q_value_pgs", np.nan)
        if not np.isfinite(q_only) or not np.isfinite(q_joint):
            return "no_signal"
        only_sig = q_only < alpha
        joint_sig = q_joint < alpha
        if only_sig and joint_sig:
            return "independent"
        if only_sig and not joint_sig:
            return "mediated"
        if not only_sig and joint_sig:
            return "suppressor"
        return "no_signal"
    
    
    def _regime_color(g):
        if not isinstance(g, str):
            return "#bdc3c7"
        return REGIME_COLORS.get(g, "#bdc3c7")
    
    
    try:
        pgs_df = pd.concat([load_pgs(lm) for lm in LANDMARKS], ignore_index=True)
    except FileNotFoundError as e:
        print(f"cox_pgs_adjusted.csv missing — skipping Figure 4 (file: {e.filename})")
    else:
        pgs_df["regime"] = pgs_df.apply(classify_regime, axis=1)
    
        # Stable PGS order across panels: by max |coef_pgs_only| across all rows.
        ordering = (pgs_df.assign(abs_pgs=pgs_df["coef_pgs_only"].abs())
                           .groupby("pgs")["abs_pgs"].max()
                           .sort_values(ascending=True))
        pgs_order = list(ordering.index)
    
        target_labs = ["Testosterone", "PSA"]
    
        fig, axes = plt.subplots(
            len(target_labs), len(LANDMARKS),
            figsize=(13, 4.5 * len(target_labs) + 1),
            sharex=True, sharey=True,
        )
        if len(target_labs) == 1:
            axes = np.array([axes])
    
        for r, lab in enumerate(target_labs):
            for c, lm in enumerate(LANDMARKS):
                ax = axes[r, c]
                sub = pgs_df.loc[
                    (pgs_df["lab_name"] == lab) & (pgs_df["landmark_days"] == lm)
                ].copy()
                if sub.empty:
                    ax.text(0.5, 0.5, "(no rows)", ha="center", va="center",
                            transform=ax.transAxes, color="#7f8c8d")
                    ax.set_axis_off()
                    continue
    
                sub = sub.set_index("pgs").reindex(pgs_order).reset_index()
                y_pos = np.arange(len(sub))
    
                # PGS-only HR (open circle, faded) + 95% CI
                only_hr = sub["hazard_ratio_pgs_only_per_sd"].to_numpy(dtype=float)
                ax.errorbar(
                    only_hr, y_pos,
                    xerr=[
                        np.clip(only_hr - sub["ci_lower_pgs_only"].to_numpy(dtype=float), 0, None),
                        np.clip(sub["ci_upper_pgs_only"].to_numpy(dtype=float) - only_hr, 0, None),
                    ],
                    fmt="o",
                    mfc="white", mec="#5d6d7e",
                    ecolor="#bdc3c7",
                    ms=6, elinewidth=1.0, capsize=0,
                    zorder=2,
                )
    
                # Joint PGS HR (filled diamond, colored by regime) + 95% CI
                joint_hr = sub["hazard_ratio_pgs_per_sd"].to_numpy(dtype=float)
                ax.errorbar(
                    joint_hr, y_pos,
                    xerr=[
                        np.clip(joint_hr - sub["ci_lower_pgs"].to_numpy(dtype=float), 0, None),
                        np.clip(sub["ci_upper_pgs"].to_numpy(dtype=float) - joint_hr, 0, None),
                    ],
                    fmt="none",
                    ecolor="#5d6d7e",
                    elinewidth=1.0, capsize=0,
                    zorder=3,
                )
                colors = [_regime_color(g) for g in sub["regime"]]
                ax.scatter(
                    joint_hr, y_pos,
                    marker="D", s=55,
                    c=colors,
                    edgecolor="white", linewidth=0.7,
                    zorder=4,
                )
    
                # Connector showing attenuation direction
                for i, (o, j) in enumerate(zip(only_hr, joint_hr)):
                    if np.isfinite(o) and np.isfinite(j):
                        ax.plot([o, j], [i, i], color="#bdc3c7",
                                linewidth=0.8, zorder=1)
    
                ax.axvline(1.0, color="grey", linestyle=":", linewidth=0.9)
                ax.set_xscale("log")
                ax.set_yticks(y_pos)
                ax.set_yticklabels(sub["pgs"], fontsize=8)
                sign = "+" if lm > 0 else ""
                ax.set_title(f"{lab}  ·  {sign}{lm} days", fontsize=11, weight="bold")
                if r == len(target_labs) - 1:
                    ax.set_xlabel("HR per SD (log scale)")
    
        # Legend: PGS-only marker + four regime diamonds
        legend_handles = [
            Line2D([0], [0], marker="o", color="w",
                   markerfacecolor="white", markeredgecolor="#5d6d7e",
                   markersize=8, linewidth=0,
                   label="PGS-only HR"),
        ]
        for regime in REGIME_ORDER:
            legend_handles.append(
                Line2D([0], [0], marker="D", color="w",
                       markerfacecolor=REGIME_COLORS[regime],
                       markeredgecolor="white",
                       markersize=8, linewidth=0,
                       label=f"joint HR — {regime}"),
            )
        fig.legend(handles=legend_handles, loc="lower center",
                   ncol=len(legend_handles), bbox_to_anchor=(0.5, -0.02),
                   fontsize=9)
        fig.suptitle("PGS interpretation — platinum endpoint",
                     fontsize=13, weight="bold", y=1.0)
        fig.tight_layout(rect=[0, 0.03, 1, 0.98])
    
        for ext in ("png", "pdf"):
            out = OUT_DIR / f"pgs_interpretation_platinum.{ext}"
            fig.savefig(out)
            print(f"wrote {out}")
        plt.show()
    
        # Regime counts per (target lab × landmark) for a quick numeric summary.
        print("\nRegime counts by (target lab × landmark):")
        summary = (
            pgs_df.groupby(["lab_name", "landmark_days", "regime"]).size()
                  .unstack("regime", fill_value=0)
                  .reindex(columns=REGIME_ORDER, fill_value=0)
        )
        print(summary)